# Mitsui-style recursive feature forecasting with TFTS

from: https://www.kaggle.com/competitions/mitsui-commodity-prediction-challenge/writeups/artem777-in-mitsui-6th-Place

1. **Recursive full-feature forecasting** — an LSTM reads a window of *all* feature
   columns and predicts the **next single feature row**. That row is fed back into the
   model and the window is re-slid, iterating `MAX_LAG + 1 = 5` times.
2. **Target construction** — each forecasted feature row is turned into the competition
   targets via a lookup table of `(a, b, lag)` pairs, computing the *log-return*
   `log(a_{t+lag}/a_{t+1})` (minus `log(b_{t+lag}/b_{t+1})` when `b` is present).
3. **Label-lag correction** — the model targets are blended with the mean of recent real
   label lags: `corrected = 0.7 * model_targets + 0.3 * lag_mean`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense

from tfts.data import AutoPreprocessor
from tfts.generation import GenerationEngine, MeanSampler, StepOutput
from tfts.models.base import BaseConfig, BaseModel

## 1. Imagine the Mitsui data

Build a synthetic matrix with the same structure as the competition:

- `train`          : one row per `date_id`, one column per feature (424 in the real
                     competition, 20 here). Geometric random walks keep every value
                     strictly positive so the log-return targets are well-defined.
- `target_pairs`   : the `(a, b, lag)` pairs that define the 424 targets. We use a small
                     hand-written subset covering both single-series and spread targets.
- `label_lags`     : the API-provided *recent real* feature rows used by the correction.


In [ ]:
rng = np.random.default_rng(42)

FEATURES = 20
N_DAYS = 600
WINDOW_SIZE = 50
MAX_LAG = 4
FORECAST_STEPS = MAX_LAG + 1


def commodity_walk(n, f, scale=0.008):
    t = np.arange(n)
    seasonal = 0.004 * np.sin(t / 17.0 + f) + 0.002 * np.cos(t / 5.0 + f)
    return np.cumsum(seasonal + rng.normal(0.0, scale, n))


feature_names = [f"feature_{i}" for i in range(FEATURES)]
train = pd.DataFrame({name: commodity_walk(N_DAYS, i) for i, name in enumerate(feature_names)})

# (a, b, lag): lag=0 -> return from t+1 to t+1 (b ignored), lag>=1 -> spread log-return.


def _pairs():
    ps = []
    for i in range(8):
        ps.append({"target": i, "lag": int(rng.integers(1, MAX_LAG + 1)), "a": feature_names[i], "b": None})
    for j in range(8, 12):
        ps.append(
            {
                "target": j,
                "lag": int(rng.integers(1, MAX_LAG + 1)),
                "a": feature_names[j],
                "b": feature_names[(j + 3) % FEATURES],
            }
        )
    return ps


target_pairs = _pairs()

# Recent *real* history (the API label lags): last 4 days, slightly corrupted.
label_lags = train.iloc[-WINDOW_SIZE : -WINDOW_SIZE + MAX_LAG + 1].copy()
for _ in range(3):
    label_lags = pd.concat([label_lags, train.iloc[-1:]], ignore_index=True)  # stand-in rows, real API gives 4 batches

train.head()

## 2. Preprocess with `AutoPreprocessor`

TFTS `AutoPreprocessor` does the same job as Mitsui's `StandardScaler` +
`fillna`: impute missing values and standardize every feature column. Keep the fitted
preprocessor around so the recursive forecasts can be mapped back to the original scale.


In [ ]:
pre = AutoPreprocessor(handle_missing="interpolate", normalize="standard", columns=feature_names)
train_scaled = pre.fit_transform(train)
train_scaled.head()

## 3. Full-feature LSTM + independent rollout engine

This is the heart of the strategy. Two ingredients:

- A small `BaseModel` whose `__call__` emits the **entire next feature row**
  (dimension `FEATURES`), exactly like Mitsui's `FeatLSTM`.
- A `GenerationEngine` whose feedback function appends the predicted full row to the
  rolling window. The model stays independent from inference policy.


In [ ]:
class FullFeatureLSTMConfig(BaseConfig):
    model_type = "full_feature_lstm"

    def __init__(self, hidden_size: int = 64, input_dim: int = FEATURES):
        super().__init__()
        self.hidden_size = hidden_size
        self.input_dim = input_dim


class FullFeatureLSTM(BaseModel):
    "Predict the whole next feature row, then roll it back for the next step."

    def __init__(self, config=None):
        super().__init__(predict_sequence_length=1, config=config or FullFeatureLSTMConfig())
        self.lstm = LSTM(self.config.hidden_size, return_sequences=False)
        self.head = Dense(self.config.input_dim)

    def call(self, inputs, training=None):
        # Emit one full feature row: [batch, 1, FEATURES]. Keeping the horizon axis
        # explicit so an external rollout can append it to the window.
        _, encoder_feature, _ = self._prepare_3d_inputs(inputs)
        hidden = self.lstm(encoder_feature)  # (batch, hidden)
        return self.head(hidden)[:, None, :]  # (batch, 1, FEATURES)

### Train

Mitsui trains the LSTM to predict the whole next feature row (`y = X[t+1]`) from a past
window (`X[t-WINDOW:t]`). This is a plain supervised/teacher-forced fit — the recursion
happens later at inference time via `generate()`.


In [ ]:
def prepare_windows(df, window_size):
    X = df.values.astype(np.float32)
    X_wins, y_next = [], []
    for i in range(len(X) - window_size):
        X_wins.append(X[i : i + window_size])
        y_next.append(X[i + window_size])
    return np.stack(X_wins), np.stack(y_next)


X_wins, y_next = prepare_windows(train_scaled, WINDOW_SIZE)
N = len(X_wins)
val_size = int(0.15 * N)
X_train, y_train = X_wins[:-val_size], y_next[:-val_size]
X_val, y_val = X_wins[-val_size:], y_next[-val_size:]
y_train = y_train[:, None, :]  # (N, 1, FEATURES) -- model emits one full next row
y_val = y_val[:, None, :]
print("train windows:", X_train.shape, "target row:", y_train.shape)

model = FullFeatureLSTM(FullFeatureLSTMConfig(hidden_size=64, input_dim=FEATURES))

keras_model = model.build_model(tf.keras.Input(shape=(WINDOW_SIZE, FEATURES)))
keras_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")

history = keras_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    batch_size=64,
    epochs=15,
    verbose=1,
)

## 4. Recursive multi-day forecast

Initialize the rolling window with the most recent observed (scaled) `WINDOW_SIZE` rows,
then ask the model to generate `FORECAST_STEPS` future *full feature rows* one day at a
time. This is the exact recursion of Mitsui's
`forecast_next_days_from_history`: predict a row, feed it back, re-slide the window.

Back-transform the generated rows to the original scale with the fitted
`AutoPreprocessor` before computing the targets.


In [ ]:
history_window = train_scaled.values[-WINDOW_SIZE:][None, ...].astype(np.float32)  # (1, WINDOW, FEATURES)


def step_fn(current, state, step):
    return StepOutput(prediction=model(current, training=False), state=state)


def feedback(current, result, **kwargs):
    return tf.concat([current[:, 1:, :], result.value], axis=1)


out = GenerationEngine(MeanSampler(), feedback).run(step_fn, history_window, initial_state=None, horizon=FORECAST_STEPS)

feat_scaled = out.values[0].numpy()  # (FORECAST_STEPS, FEATURES)
feat_forecast = pre.inverse_transform(pd.DataFrame(feat_scaled, columns=feature_names)).values
print("forecasted feature rows:", feat_forecast.shape)
feat_forecast

## 5. From feature forecasts to competition targets

Replicate `compute_targets_from_feature_forecasts`: for each `(a, b, lag)` pair, take
the forecasted row at `t+1` and at `t+lag`, and form the (spread) log-return.


In [ ]:
def compute_targets(forecasts, pairs, cols):
    idx = {c: i for i, c in enumerate(cols)}
    results = []
    for p in pairs:
        a, b, lag = p["a"], p["b"], p["lag"]
        if a not in idx:
            results.append(0.0)
            continue
        a_t1 = max(forecasts[0][idx[a]], 1e-9)
        a_tlag = max(forecasts[min(lag, len(forecasts) - 1)][idx[a]], 1e-9)
        a_ret = np.log(a_tlag / a_t1)
        if b is None or b not in idx:
            results.append(a_ret)
        else:
            b_t1 = max(forecasts[0][idx[b]], 1e-9)
            b_tlag = max(forecasts[min(lag, len(forecasts) - 1)][idx[b]], 1e-9)
            results.append(a_ret - np.log(b_tlag / b_t1))
    return np.array(results, dtype=np.float32)


model_targets = compute_targets(feat_forecast, target_pairs, feature_names)
print("raw model targets:", np.round(model_targets, 4))

## 6. Label-lag correction (0.7 / 0.3 blend)

Finally blunt the model's drift away from the current market state by blending with the
mean of the recent real label lags, exactly as Mitsui does:

```python
corrected = alpha * model_targets + (1 - alpha) * correction
alpha = 0.7
```

The "correction" for a pair is the recent-observed mean of `a - b` (falling back to `a`
or `-b` if one series is missing), built from the API-provided `label_lags` batches.


In [ ]:
alpha = 0.7

# Mean of the recent real feature rows, indexed by feature name (like Mitsui's lag_mean).
lag_mean = label_lags.mean(axis=0).fillna(0.0)

results = pd.DataFrame(
    {
        "a": [p["a"] for p in target_pairs],
        "b": [p["b"] for p in target_pairs],
        "lag": [p["lag"] for p in target_pairs],
        "model": model_targets,
        "lag_mean": [lag_mean[p["a"]] - (lag_mean[p["b"]] if p["b"] in lag_mean else 0.0) for p in target_pairs],
    }
)
results["corrected"] = alpha * results["model"] + (1 - alpha) * results["lag_mean"]
results

## 7. Sanity check

A corrected target should sit between the raw model target and the lag-mean anchor —
i.e. it dampens large model swings towards the recent market state, which is precisely
the intended effect during turbulent periods.


In [ ]:
spreads = results.iloc[8:]  # the (a - b) spread targets, where the shift is most visible
fig, ax = plt.subplots(figsize=(8, 3.2))

x = np.arange(len(spreads))
ax.plot(x, spreads["model"], "o--", color="#1f77b4", label="model only (0.7 implied)")
ax.plot(x, spreads["corrected"], "s--", color="#d62728", label="corrected (0.7/0.3 blend)")
ax.plot(x, spreads["lag_mean"], "d:", color="#7f7f7f", label="lag-mean anchor (0.3)")
ax.axhline(0.0, color="k", lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(
    [f'{r["a"].replace("feature_", "f")}-{r["b"].replace("feature_", "f")}' for _, r in spreads.iterrows()],
    rotation=45,
    ha="right",
)
ax.set_ylabel("spread log-return target")
ax.set_title("Label-lag correction pulls raw forecasts toward the recent market state")
ax.legend(frameon=False, ncol=3)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()